# UNSW-NB15 Baseline Models

**Flow:** load preprocessed data -> train Random Forest, LightGBM, XGBoost, Neural Network -> evaluate -> compare.

All four models train on `train_smote.parquet` (SMOTE-floored at 5000 rows per class, see the preprocessing notebook) and are evaluated on `test.parquet`, which keeps the real class imbalance and is never touched during training or tuning. A 15% slice of the training data is held out as a validation set for early stopping and the loss curves below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import lightgbm as lgb
import xgboost as xgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load data

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"

train_df = pd.read_parquet(f"{DATASET_PATH}/train_smote.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
meta = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")
class_names = meta["label_encoder"].classes_
n_classes = len(class_names)

feature_cols = [c for c in train_df.columns if c != "label"]
X_train_full, y_train_full = train_df[feature_cols], train_df["label"]
X_test, y_test = test_df[feature_cols], test_df["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, stratify=y_train_full, random_state=SEED
)

print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("classes:", list(class_names))

SMOTE only floors the five smallest classes (see the preprocessing notebook) - Normal, Exploits, Fuzzers, Generic and Reconnaissance are untouched, so the training set is still imbalanced. Per-class weights are computed once here and reused by every model below.

In [ ]:
class_weight_arr = compute_class_weight("balanced", classes=np.arange(n_classes), y=y_train)
class_weight_dict = dict(enumerate(class_weight_arr))
print({class_names[i]: round(w, 2) for i, w in class_weight_dict.items()})

## 2. Shared evaluation helper

Same function for every model so the results are directly comparable: accuracy, macro F1, a full classification report, and a row-normalized confusion matrix.

In [ ]:
def evaluate(name, y_true, y_pred):
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    print(f"{name} | accuracy: {acc:.4f} | macro F1: {macro_f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    cm = confusion_matrix(y_true, y_pred, normalize="true")
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names,
                cbar_kws={"label": "share of true class"})
    plt.xlabel("predicted")
    plt.ylabel("true")
    plt.title(f"{name} - confusion matrix (row-normalized, test set)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return {"model": name, "accuracy": acc, "macro_f1": macro_f1}


results = []

## 3. Random Forest

Trees are added in steps with `warm_start=True` so validation macro F1 can be tracked as the forest grows - the closest thing a Random Forest has to a training curve, since it has no iterative loss.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=0, max_depth=20, class_weight="balanced_subsample",
    warm_start=True, n_jobs=-1, random_state=SEED,
)

tree_steps = [25, 50, 100, 150, 200]
rf_curve = []
for n in tree_steps:
    rf.n_estimators = n
    rf.fit(X_train, y_train)
    val_f1 = f1_score(y_val, rf.predict(X_val), average="macro")
    rf_curve.append(val_f1)
    print(f"n_estimators={n:>3} | val macro F1: {val_f1:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(tree_steps, rf_curve, marker="o")
plt.xlabel("n_estimators")
plt.ylabel("val macro F1")
plt.title("Random Forest - validation macro F1 vs. number of trees")
plt.tight_layout()
plt.show()

In [ ]:
rf_pred = rf.predict(X_test)
results.append(evaluate("Random Forest", y_test, rf_pred))

imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
plt.figure(figsize=(7, 5))
imp.sort_values().plot.barh(color="steelblue")
plt.xlabel("importance")
plt.title("Random Forest - top 15 features")
plt.tight_layout()
plt.show()

## 4. LightGBM

Native multiclass gradient boosting. `valid_sets` tracks multi-class logloss on train and validation every round, giving a real training/loss curve, with early stopping to avoid overfitting.

In [ ]:
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

lgb_params = {
    "objective": "multiclass", "num_class": n_classes, "metric": "multi_logloss",
    "class_weight": "balanced", "learning_rate": 0.08, "num_leaves": 63,
    "seed": SEED, "verbose": -1,
}

lgb_evals = {}
lgb_model = lgb.train(
    lgb_params, lgb_train, num_boost_round=300, valid_sets=[lgb_train, lgb_val], valid_names=["train", "val"],
    callbacks=[lgb.record_evaluation(lgb_evals), lgb.early_stopping(20), lgb.log_evaluation(0)],
)

plt.figure(figsize=(7, 4))
plt.plot(lgb_evals["train"]["multi_logloss"], label="train")
plt.plot(lgb_evals["val"]["multi_logloss"], label="val")
plt.xlabel("boosting round")
plt.ylabel("multi logloss")
plt.legend()
plt.title("LightGBM - training/validation loss curve")
plt.tight_layout()
plt.show()

In [ ]:
lgb_pred = np.argmax(lgb_model.predict(X_test), axis=1)
results.append(evaluate("LightGBM", y_test, lgb_pred))

imp = pd.Series(lgb_model.feature_importance(importance_type="gain"), index=feature_cols).sort_values(ascending=False).head(15)
plt.figure(figsize=(7, 5))
imp.sort_values().plot.barh(color="darkorange")
plt.xlabel("gain")
plt.title("LightGBM - top 15 features")
plt.tight_layout()
plt.show()

## 5. XGBoost

Second gradient-boosting baseline for comparison with LightGBM. Same idea: an eval set with `mlogloss` gives the loss curve, early stopping picks the best round.

In [ ]:
sample_weight = class_weight_arr[y_train.values]

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
    n_estimators=300, learning_rate=0.08, max_depth=8,
    early_stopping_rounds=20, random_state=SEED, n_jobs=-1,
)
xgb_model.fit(
    X_train, y_train, sample_weight=sample_weight,
    eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False,
)

evals = xgb_model.evals_result()
plt.figure(figsize=(7, 4))
plt.plot(evals["validation_0"]["mlogloss"], label="train")
plt.plot(evals["validation_1"]["mlogloss"], label="val")
plt.xlabel("boosting round")
plt.ylabel("mlogloss")
plt.legend()
plt.title("XGBoost - training/validation loss curve")
plt.tight_layout()
plt.show()

In [ ]:
xgb_pred = xgb_model.predict(X_test)
results.append(evaluate("XGBoost", y_test, xgb_pred))

imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
plt.figure(figsize=(7, 5))
imp.sort_values().plot.barh(color="seagreen")
plt.xlabel("importance")
plt.title("XGBoost - top 15 features")
plt.tight_layout()
plt.show()

## 6. Neural network

Small MLP: two hidden layers with batch norm and dropout, softmax output over the 10 classes. The class weights from step 1 go straight into the loss so the rare classes aren't ignored. Early stopping on validation loss avoids picking an overfit epoch.

In [ ]:
nn_model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(64, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(n_classes, activation="softmax"),
])
nn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = nn_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=30, batch_size=2048, class_weight=class_weight_dict,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    verbose=0,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Neural network - loss curve")
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
axes[1].set_title("Neural network - accuracy curve")
plt.tight_layout()
plt.show()

In [ ]:
nn_pred = np.argmax(nn_model.predict(X_test, verbose=0), axis=1)
results.append(evaluate("Neural Network", y_test, nn_pred))

## 8. Two-stage classification

Normal (389,780 rows) dominates the test set and dilutes every single-model score above. Splitting the problem in two removes that dilution:

1. **Stage 1 (binary):** attack vs. Normal.
2. **Stage 2 (multiclass):** only on rows Stage 1 calls "attack", classify which of the 9 attack types it is.

Both stages use LightGBM (the strongest single model above). A row only reaches Stage 2 if Stage 1 flags it as an attack - a Stage 1 miss means it's scored as Normal regardless of its real type, same as any other misclassification.

In [ ]:
normal_label = list(class_names).index("Normal")
attack_label_ids = [i for i in range(n_classes) if i != normal_label]
attack_classes = class_names[attack_label_ids]
remap = {orig: new for new, orig in enumerate(attack_label_ids)}   # LightGBM needs contiguous 0..n-1 labels

y_train_bin = (y_train != normal_label).astype(int)
y_val_bin = (y_val != normal_label).astype(int)

stage1 = lgb.LGBMClassifier(objective="binary", class_weight="balanced", learning_rate=0.08,
                             num_leaves=63, n_estimators=300, random_state=SEED, verbose=-1)
stage1.fit(X_train, y_train_bin, eval_set=[(X_val, y_val_bin)],
           callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)])

attack_train, attack_val = y_train != normal_label, y_val != normal_label
y_train_attack = y_train[attack_train].map(remap)
y_val_attack = y_val[attack_val].map(remap)

stage2 = lgb.LGBMClassifier(objective="multiclass", num_class=len(attack_classes), class_weight="balanced",
                             learning_rate=0.08, num_leaves=63, n_estimators=300, random_state=SEED, verbose=-1)
stage2.fit(X_train[attack_train], y_train_attack, eval_set=[(X_val[attack_val], y_val_attack)],
           callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)])

print("stage 1 (binary) and stage 2 (attack type) trained")

In [ ]:
stage1_pred = stage1.predict(X_test)

combined_pred = np.full(len(X_test), normal_label)
attack_idx = np.where(stage1_pred == 1)[0]
stage2_pred = stage2.predict(X_test.iloc[attack_idx])
combined_pred[attack_idx] = np.array(attack_label_ids)[stage2_pred]

print(f"stage 1 flagged {len(attack_idx)} of {len(X_test)} rows as attack")
results.append(evaluate("Two-Stage (LightGBM)", y_test, combined_pred))

## 9. Ensemble

Soft-voting average of the three tree models' class probabilities (Random Forest, LightGBM, XGBoost). The Neural Network is left out - it's the weakest and least-calibrated of the four, and averaging it in tends to drag the ensemble down rather than help.

In [ ]:
proba_rf = rf.predict_proba(X_test)
proba_lgb = lgb_model.predict(X_test)
proba_xgb = xgb_model.predict_proba(X_test)

ensemble_proba = (proba_rf + proba_lgb + proba_xgb) / 3
ensemble_pred = np.argmax(ensemble_proba, axis=1)

results.append(evaluate("Ensemble (RF+LightGBM+XGBoost)", y_test, ensemble_pred))

In [ ]:
# Save test-set probabilities (indexed by row_id) so the GNN notebook can fuse them with its own.
# Two-stage as probabilities: P(Normal) = 1 - P(attack), P(attack type) = P(attack) * P(type | attack).
p_attack = stage1.predict_proba(X_test)[:, 1]
two_stage_proba = np.zeros((len(X_test), n_classes), dtype="float32")
two_stage_proba[:, normal_label] = 1 - p_attack
two_stage_proba[:, attack_label_ids] = p_attack[:, None] * stage2.predict_proba(X_test)

for name, proba in [("two_stage", two_stage_proba), ("ensemble", ensemble_proba)]:
    pd.DataFrame(proba, index=X_test.index, columns=list(class_names)).to_parquet(f"/kaggle/working/{name}_test_proba.parquet")

print("two-stage (soft probabilities) macro F1:", round(f1_score(y_test, two_stage_proba.argmax(1), average="macro"), 4))

## 10. Model comparison

All six (4 single models + two-stage + ensemble) evaluated on the same held-out `test.parquet` (real class balance, never used for training or early stopping).

In [ ]:
results_df = pd.DataFrame(results).set_index("model")
display(results_df.round(4))

results_df[["accuracy", "macro_f1"]].plot.bar(figsize=(7, 4), color=["#4C72B0", "#DD8452"])
plt.ylabel("score")
plt.title("Baseline comparison - accuracy vs macro F1 (test set)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Macro F1 is the number that matters here - it treats all 10 classes equally, so it reflects how well each model does on the rare attack types (Worms, Shellcode, Backdoor, Analysis, DoS), not just on Normal/Exploits which dominate the row count. This table is the baseline the GNN needs to beat.